# 01 — Kiểm kê và gán nhãn từng gói cho data capture

Theo `PLAN_tron_nen_window.md` mục 4–5.

- Đọc metadata 1.682 task (11 kỹ thuật recon + SSH brute force).
- Tách bảng gói từ pcap bằng `tshark`.
- Gán nhãn **từng gói** theo luật IP/MAC/port/cờ của kỹ thuật (không theo thư mục, không theo `status`):
  `attack`, `aux`, `artifact`, `laptop_other`, `unknown`.
- Đầu ra: `Data_after_processing/attack_packets/<scenario>/task_XXXX.parquet` và báo cáo `reports/01_*.csv`.

In [1]:
import json, subprocess
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd

ROOT = Path('/home/ubuntu/sepcung/03.Brute_Force_vs_Recon')
RAW = ROOT / 'Data' / 'real_data'
OUT = ROOT / 'Data_after_processing'
PKT_DIR = OUT / 'attack_packets'
REPORT_DIR = OUT / 'reports'
PKT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

ATTACKER_IP = '192.168.1.236'
ATTACKER_MAC = '50:bb:b5:fe:d8:b8'
TARGET_IP = '192.168.1.182'
ROUTER_IP = '192.168.1.1'
LAN_PREFIX = '192.168.1.'

N_JOBS = 32
MIN_ATTACK_PKTS = 5      # task có ít hơn số gói attack này -> không dùng
MAX_DROP_RATE = 0.20     # kernel_drops / packets_received vượt ngưỡng -> đánh dấu

# kịch bản -> nhóm luật
SCENARIO_RULE = {
    'tcp_syn_scan': 'port_scan', 'tcp_connect_scan': 'port_scan',
    'stealth_fin_scan': 'port_scan', 'stealth_null_scan': 'port_scan',
    'stealth_xmas_scan': 'port_scan', 'firewall_ack_mapping': 'port_scan',
    'udp_port_scan': 'port_scan',
    'icmp_echo_discovery': 'icmp_discovery',
    'tcp_ack_host_discovery': 'tcp443_discovery', 'tcp_syn_host_discovery': 'tcp443_discovery',
    'arp_discovery': 'arp_discovery',
    'ssh': 'ssh_bruteforce',
}
SCENARIO_CLASS = {s: ('bruteforce' if r == 'ssh_bruteforce' else 'recon') for s, r in SCENARIO_RULE.items()}

## 1. Kiểm kê task từ metadata

In [2]:
rows = []
for scen_dir in sorted(list((RAW / 'recon').iterdir()) + [RAW / 'brute_force' / 'ssh']):
    if not (scen_dir / 'metadata').is_dir():
        continue
    scen = scen_dir.name
    for mf in sorted((scen_dir / 'metadata').glob('task_*.json')):
        m = json.loads(mf.read_text(encoding='utf-8-sig'))
        q = m.get('quality') or {}
        td = q.get('tcpdump') or {}
        pcap = scen_dir / 'pcaps' / (mf.stem + '.pcap')
        rows.append({
            'scenario': scen,
            'rule': SCENARIO_RULE[scen],
            'class': SCENARIO_CLASS[scen],
            'task_id': m.get('task_id'),
            'technique': m.get('technique'),
            'tool': m.get('tool'),
            'status': m.get('status'),
            'issues': '|'.join(q.get('issues') or []),
            'packets_captured': td.get('packets_captured'),
            'packets_received': td.get('packets_received'),
            'kernel_drops': td.get('kernel_drops'),
            'run_id': m.get('run_id'),
            'command': m.get('command'),
            'pcap': str(pcap),
            'pcap_exists': pcap.exists(),
        })
inv = pd.DataFrame(rows)
inv['drop_rate'] = (inv['kernel_drops'].fillna(0) / inv['packets_received'].replace(0, np.nan)).fillna(0)
print(len(inv), 'task;', inv['pcap_exists'].sum(), 'pcap tồn tại')
inv.groupby('scenario').agg(tasks=('task_id', 'size'),
                            completed=('status', lambda s: (s == 'COMPLETED').sum()),
                            with_drops=('kernel_drops', lambda s: (s.fillna(0) > 0).sum()))

1681 task; 1681 pcap tồn tại


,tasks,completed,with_drops
scenario,,,
arp_discovery,68,0,67
firewall_ack_mapping,256,50,10
icmp_echo_discovery,136,136,0
ssh,283,52,2
stealth_fin_scan,32,16,0
stealth_null_scan,32,15,0
stealth_xmas_scan,32,15,0
tcp_ack_host_discovery,24,3,21
tcp_connect_scan,29,6,17


## 2. Tách bảng gói và gán nhãn

Luật (mục 5 của plan):
- `artifact`: TCP port 22 giữa `.236` và `.1` — luồng plink stream pcap về laptop, khi deploy không tồn tại.
- `attack` theo nhóm luật:
  - `port_scan`: mọi gói IP giữa `.236` và `.182` (kể cả ICMP unreachable trả lời UDP scan).
  - `icmp_discovery`: ICMP echo request từ `.236` tới IP LAN, echo reply từ IP LAN về `.236`.
  - `tcp443_discovery`: TCP port 443 giữa `.236` và IP LAN (probe, RST/SYN-ACK trả lời, RST của laptop).
  - `arp_discovery`: ARP request có MAC gửi là attacker; ARP reply gửi tới MAC attacker.
  - `ssh_bruteforce`: TCP port 22 giữa `.236` và `.182`.
- `aux`: ARP của attacker ở các kỹ thuật khác (phân giải địa chỉ trước khi quét).
- `laptop_other`: traffic còn lại của laptop (Internet, DNS…).
- `unknown`: còn lại.

In [3]:
FIELDS = ['frame.time_epoch', 'frame.len', 'eth.src', 'eth.dst', 'ip.src', 'ip.dst', 'ip.proto', 'ip.ttl',
          'tcp.srcport', 'tcp.dstport', 'tcp.flags', 'tcp.len', 'udp.srcport', 'udp.dstport', 'icmp.type',
          'arp.opcode', 'arp.src.hw_mac', 'arp.src.proto_ipv4', 'arp.dst.proto_ipv4']
COLS = ['ts', 'len', 'eth_src', 'eth_dst', 'ip_src', 'ip_dst', 'ip_proto', 'ttl',
        'tcp_sport', 'tcp_dport', 'tcp_flags', 'tcp_len', 'udp_sport', 'udp_dport', 'icmp_type',
        'arp_op', 'arp_sha', 'arp_spa', 'arp_tpa']
INT_COLS = ['len', 'ip_proto', 'ttl', 'tcp_sport', 'tcp_dport', 'tcp_len', 'udp_sport', 'udp_dport', 'icmp_type', 'arp_op']


def read_pcap(pcap):
    cmd = ['tshark', '-r', pcap, '-n', '-T', 'fields', '-E', 'separator=\t', '-E', 'occurrence=f']
    for f in FIELDS:
        cmd += ['-e', f]
    res = subprocess.run(cmd, capture_output=True, text=True)
    lines = [l.split('\t') for l in res.stdout.splitlines() if l]
    df = pd.DataFrame(lines, columns=COLS) if lines else pd.DataFrame(columns=COLS)
    df['ts'] = pd.to_numeric(df['ts'], errors='coerce')
    for c in INT_COLS:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int32')
    df['tcp_flags'] = df['tcp_flags'].map(lambda x: int(x, 16) if x else pd.NA).astype('Int32')
    # ip.proto lấy lớp ngoài: cổng TCP/UDP bên trong ICMP error không được tính là gói TCP/UDP
    is_icmp = df['ip_proto'] == 1
    df.loc[is_icmp, ['tcp_sport', 'tcp_dport', 'tcp_flags', 'tcp_len', 'udp_sport', 'udp_dport']] = pd.NA
    for c in ['eth_src', 'eth_dst', 'arp_sha']:
        df[c] = df[c].str.lower()
    return df


def label_packets(df, rule):
    src, dst = df['ip_src'], df['ip_dst']
    is_arp = df['arp_op'].notna()
    is_tcp = df['ip_proto'] == 6
    is_icmp = df['ip_proto'] == 1
    tport = lambda p: is_tcp & ((df['tcp_sport'] == p) | (df['tcp_dport'] == p))
    pair = lambda a, b: ((src == a) & (dst == b)) | ((src == b) & (dst == a))
    lan = lambda s: s.fillna('').str.startswith(LAN_PREFIX)
    attacker_ip = (src == ATTACKER_IP) | (dst == ATTACKER_IP)
    attacker_arp = is_arp & ((df['arp_sha'] == ATTACKER_MAC) | (df['eth_dst'] == ATTACKER_MAC)
                             | (df['arp_spa'] == ATTACKER_IP) | (df['arp_tpa'] == ATTACKER_IP))
    lan_peer = ((src == ATTACKER_IP) & lan(dst)) | ((dst == ATTACKER_IP) & lan(src))

    if rule == 'port_scan':
        attack = pair(ATTACKER_IP, TARGET_IP)
    elif rule == 'ssh_bruteforce':
        attack = pair(ATTACKER_IP, TARGET_IP) & tport(22)
    elif rule == 'icmp_discovery':
        attack = is_icmp & (((src == ATTACKER_IP) & lan(dst) & (df['icmp_type'] == 8))
                            | ((dst == ATTACKER_IP) & lan(src) & (df['icmp_type'] == 0)))
    elif rule == 'tcp443_discovery':
        attack = lan_peer & tport(443)
    elif rule == 'arp_discovery':
        attack = is_arp & (((df['arp_op'] == 1) & (df['arp_sha'] == ATTACKER_MAC))
                           | ((df['arp_op'] == 2) & (df['eth_dst'] == ATTACKER_MAC)))
    else:
        raise ValueError(rule)

    artifact = pair(ATTACKER_IP, ROUTER_IP) & tport(22)
    other = attacker_ip | (df['eth_src'] == ATTACKER_MAC) | (df['eth_dst'] == ATTACKER_MAC)
    conds = [c.fillna(False).to_numpy(dtype=bool) for c in [artifact, attack, attacker_arp, other]]
    lab = np.select(conds, ['artifact', 'attack', 'aux', 'laptop_other'], default='unknown')
    df['label'] = lab
    return df


LABELS = ['attack', 'aux', 'artifact', 'laptop_other', 'unknown']


def process_task(row):
    df = read_pcap(row['pcap'])
    df = label_packets(df, row['rule'])
    out = PKT_DIR / row['scenario'] / (Path(row['pcap']).stem + '.parquet')
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out, index=False)
    att = df.loc[df['label'] == 'attack', 'ts']
    res = {'scenario': row['scenario'], 'task_id': row['task_id'], 'n_pkts': len(df),
           'pcap_t0': df['ts'].min(), 'pcap_t1': df['ts'].max(),
           'attack_t0': att.min(), 'attack_t1': att.max(), 'parquet': str(out)}
    vc = df['label'].value_counts()
    for l in LABELS:
        res['n_' + l] = int(vc.get(l, 0))
    unk = df[df['label'] == 'unknown'].head(5).assign(scenario=row['scenario'], task_id=row['task_id'])
    return res, unk

In [4]:
todo = inv[inv['pcap_exists']].to_dict('records')
with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
    results = list(ex.map(process_task, todo, chunksize=4))

stats = pd.DataFrame([r for r, _ in results])
unknown_samples = pd.concat([u for _, u in results if len(u)], ignore_index=True) if any(len(u) for _, u in results) else pd.DataFrame()
print(len(stats), 'task đã xử lý,', stats['n_pkts'].sum(), 'gói')

1681 task đã xử lý, 76537490 gói


## 3. Kiểm tra chất lượng và chọn task dùng được

In [5]:
task = inv.merge(stats, on=['scenario', 'task_id'], how='left')
task['attack_duration_s'] = task['attack_t1'] - task['attack_t0']
task['attack_pps'] = task['n_attack'] / task['attack_duration_s'].where(task['attack_duration_s'] > 0)


def usable_reason(r):
    if not r['pcap_exists']:
        return 'no_pcap'
    if r['n_attack'] < MIN_ATTACK_PKTS:
        return 'too_few_attack_pkts'
    if 'TASK_ERROR' in (r['issues'] or ''):
        return 'capture_stream_exited'
    return 'ok'


task['use_reason'] = task.apply(usable_reason, axis=1)
task['usable'] = task['use_reason'] == 'ok'
task['high_drop'] = task['drop_rate'] > MAX_DROP_RATE
task.to_csv(REPORT_DIR / '01_task_inventory.csv', index=False)

by_scen = task.groupby('scenario').agg(
    klass=('class', 'first'), tasks=('task_id', 'size'), completed=('status', lambda s: (s == 'COMPLETED').sum()),
    usable=('usable', 'sum'), usable_high_drop=('high_drop', lambda s: (s & task.loc[s.index, 'usable']).sum()),
    pkts=('n_pkts', 'sum'), attack=('n_attack', 'sum'), aux=('n_aux', 'sum'), artifact=('n_artifact', 'sum'),
    laptop_other=('n_laptop_other', 'sum'), unknown=('n_unknown', 'sum'),
    median_attack_pkts_per_usable_task=('n_attack', lambda s: s[task.loc[s.index, 'usable']].median()),
    median_attack_pps=('attack_pps', lambda s: s[task.loc[s.index, 'usable']].median()),
).reset_index()
by_scen['attack_share'] = (by_scen['attack'] / by_scen['pkts']).round(3)
by_scen.to_csv(REPORT_DIR / '01_label_counts_by_scenario.csv', index=False)
by_scen

,scenario,klass,tasks,completed,usable,usable_high_drop,pkts,attack,aux,artifact,laptop_other,unknown,median_attack_pkts_per_usable_task,median_attack_pps,attack_share
0,arp_discovery,recon,68,0,2,0,3832885,1044,0,3727298,104543,0,485.0,76.990516,0.000
1,firewall_ack_mapping,recon,256,50,252,6,136785,135777,1008,0,0,0,24.0,21.634143,0.993
2,icmp_echo_discovery,recon,136,136,118,0,4998,1417,0,0,3581,0,6.0,1.153949,0.284
3,ssh,bruteforce,283,52,282,0,61431984,61427880,4104,0,0,0,1829.5,47.227325,1.000
4,stealth_fin_scan,recon,32,16,16,0,33574,33358,216,0,0,0,1520.0,44.650062,0.994
5,stealth_null_scan,recon,32,15,15,0,39456,39243,213,0,0,0,44.0,57.497321,0.995
6,stealth_xmas_scan,recon,32,15,15,0,33370,33184,186,0,0,0,44.0,47.289974,0.994
7,tcp_ack_host_discovery,recon,24,3,22,0,1033259,637,4995,1011816,15811,0,14.5,4.294955,0.001
8,tcp_connect_scan,recon,29,6,28,12,86672,86649,23,0,0,0,2896.5,2658.912449,1.000
9,tcp_syn_host_discovery,recon,21,8,6,0,473381,88,188,472615,490,0,13.5,6.433547,0.000


In [6]:
# Task FAILED nhưng vẫn có tấn công dùng được — kiểm chứng "FAILED ≠ không có tấn công"
fail_x = (task.assign(issues=task['issues'].replace('', 'NONE'))
          .groupby(['scenario', 'status', 'issues'])
          .agg(tasks=('task_id', 'size'), usable=('usable', 'sum'), median_attack=('n_attack', 'median'))
          .reset_index())
fail_x.to_csv(REPORT_DIR / '01_status_issue_vs_usable.csv', index=False)
fail_x[fail_x['status'] == 'FAILED']

,scenario,status,issues,tasks,usable,median_attack
0,arp_discovery,FAILED,KERNEL_DROPS,33,2,2.0
1,arp_discovery,FAILED,KERNEL_DROPS|CAPTURE_COUNT_MISMATCH,5,0,2.0
2,arp_discovery,FAILED,KERNEL_DROPS|NO_EXPECTED_ATTACK_PACKETS,25,0,0.0
3,arp_discovery,FAILED,KERNEL_DROPS|NO_EXPECTED_ATTACK_PACKETS|CAPTUR...,4,0,0.0
4,arp_discovery,FAILED,MISSING_TCPDUMP_COUNTERS|NO_EXPECTED_ATTACK_PA...,1,0,0.0
6,firewall_ack_mapping,FAILED,KERNEL_DROPS,10,10,3903.5
7,firewall_ack_mapping,FAILED,NONE,192,192,19.5
8,firewall_ack_mapping,FAILED,NO_EXPECTED_ATTACK_PACKETS,4,0,0.0
11,ssh,FAILED,KERNEL_DROPS|TCP_SEQUENCE_GAPS,2,2,1594913.5
12,ssh,FAILED,MISSING_TCPDUMP_COUNTERS|TASK_ERROR: Router ca...,1,0,498.0


In [7]:
# Mẫu gói 'unknown' để kiểm tra luật
if len(unknown_samples):
    unknown_samples.to_csv(REPORT_DIR / '01_unknown_samples.csv', index=False)
    print(unknown_samples.groupby('scenario').size())
    display(unknown_samples.head(20))
else:
    print('Không có gói unknown')

Không có gói unknown


In [8]:
# Spot-check: phân bố loại gói 'attack' theo kịch bản (proto / cờ TCP / ICMP / ARP)
def sig(df):
    p = df['ip_proto'].astype('string').fillna('arp')
    f = df['tcp_flags'].map(lambda x: '' if pd.isna(x) else f'0x{int(x):02x}')
    i = df['icmp_type'].astype('string').fillna('')
    return p + '/' + f + '/' + i

spot = []
for scen, g in task[task['usable']].groupby('scenario'):
    for pq in g['parquet'].sample(min(5, len(g)), random_state=0):
        d = pd.read_parquet(pq)
        d = d[d['label'] == 'attack']
        d = d.assign(dir=np.where(d['ip_src'].eq(ATTACKER_IP) | d['eth_src'].eq(ATTACKER_MAC), 'from_attacker', 'to_attacker'))
        s = d.groupby(['dir', sig(d)]).size().rename('n').reset_index().rename(columns={0: 'sig'})
        s.columns = ['dir', 'proto/flags/icmp', 'n']
        spot.append(s.assign(scenario=scen))
spot = pd.concat(spot).groupby(['scenario', 'dir', 'proto/flags/icmp'])['n'].sum().reset_index()
spot = spot.sort_values(['scenario', 'n'], ascending=[True, False]).groupby('scenario').head(6)
spot.to_csv(REPORT_DIR / '01_attack_signature_spotcheck.csv', index=False)
spot

,scenario,dir,proto/flags/icmp,n
0,arp_discovery,from_attacker,arp//,927
1,arp_discovery,to_attacker,arp//,43
3,firewall_ack_mapping,to_attacker,6/0x04/,68
2,firewall_ack_mapping,from_attacker,6/0x10/,34
5,icmp_echo_discovery,to_attacker,1//0,21
4,icmp_echo_discovery,from_attacker,1//8,18
12,ssh,to_attacker,6/0x04/,513819
8,ssh,from_attacker,6/0x10/,231442
15,ssh,to_attacker,6/0x12/,93188
11,ssh,from_attacker,6/0x18/,49220
